# Step 2: E1 baseline

Spec E1: run **one unoptimized prompt** on test inputs and save at least 3 outputs.

Test inputs come from the BANKING77 test split (`PolyAI/banking77`). The Hub repo only holds a loading script that downloads `test.csv` from PolyAI's GitHub, so this notebook reads that same CSV directly. The `datasets` library no longer runs that script.

Each output is saved with its gold intent label to `results/e1_baseline.json`, so later steps can compare against it.

In [1]:
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3:1.7b"


def chat(messages, temperature=0.0, max_tokens=256):
    """Send chat messages to the local Ollama server and return the reply text."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "think": False,
        "options": {"temperature": temperature, "num_predict": max_tokens},
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    return body["message"]["content"].strip()

## Test inputs from BANKING77

Sample a fixed number of test rows with a fixed seed, so every run uses the same queries.

In [2]:
import csv
import io
import random

BANKING77_TEST_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
N_SAMPLES = 5
SEED = 77

with urllib.request.urlopen(BANKING77_TEST_URL, timeout=60) as resp:
    rows = list(csv.DictReader(io.StringIO(resp.read().decode("utf-8"))))

samples = random.Random(SEED).sample(rows, N_SAMPLES)
print(f"Loaded {len(rows)} test rows, sampled {len(samples)}")

Loaded 3080 test rows, sampled 5


## Unoptimized prompt

Deliberately plain: no label list, no examples, no role, no output format. This is the baseline the E2 prompt variants will be compared against.

In [3]:
BASELINE_PROMPT = "Classify the banking customer query into an intent and reply.\n\nQuery: {query}"

results = []
for i, row in enumerate(samples, start=1):
    query = row["text"]
    prompt = BASELINE_PROMPT.format(query=query)
    output = chat([{"role": "user", "content": prompt}])
    results.append({
        "id": i,
        "query": query,
        "gold_intent": row["category"],
        "prompt": prompt,
        "output": output,
    })
    print(f"--- {i} [gold: {row['category']}]\n{query}\n{output}\n")

--- 1 [gold: top_up_reverted]
Has my top up been reverted?
**Intent:** Clarification / Inquiry  
**Reply:** Please check your account statement or contact customer support to confirm if your top-up has been reverted.

--- 2 [gold: topping_up_by_card]
I have friends that would like to top-up my account is that possible?
**Intent:**  
**"Can I top up my account with friends?"**

**Reply:**  
Yes, you can top up your account using your friend's payment method. You would need to provide your bank details or use a payment gateway to link your account to your friend's card. Let me know if you need help with the process!

--- 3 [gold: card_payment_fee_charged]
I would appreciate that someone let me know when there is an extra charge for payments. I was checking the app earlier and saying a charge on one of my payments that was additional that no one made me aware of before.
**Intent:**  
**Request for Information**  
*(Specifically, the customer is asking for notification about additional cha

In [4]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
record = {
    "experiment": "E1_baseline",
    "dataset": "PolyAI/banking77 (test.csv)",
    "sample_seed": SEED,
    "model": MODEL,
    "decoding": {"do_sample": False, "temperature": 0.0, "max_tokens": 256},
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "results": results,
}
out_path = out_dir / "e1_baseline.json"
out_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(results)} outputs to {out_path}")

Saved 5 outputs to results\e1_baseline.json
